---

External tools

---

In [1]:
# autoload
%load_ext autoreload
%autoreload 2

# load relevant pgl classes
from pgl import pgl, pglDatabase, pglExternalTool, pglActions, pglFlywheel, pglAFNI, pglActionsMRI, pglActionsMNE, pglActionsFixes, pglGLMsingle, pglMATLAB, pglDcm2niix

In [ ]:
# display tools that are available for use and status
reports = pglExternalTool.print()

---

DICOM conversion with dcm2niix

---

MRI data may arrive as DICOM files. **dcm2niix** converts these files into NIfTI images and writes JSON sidecars containing scan metadata. PGL uses it when `loadFlywheel` is run with `dicom=True`.

The `pgl_mri` Conda environment includes dcm2niix. If you are using another Conda environment, you can check for it below and install it into the active environment if needed. The installer asks for confirmation before making changes. After installation, restart the notebook kernel and run the check again. If you installed dcm2niix another way, use `pglDcm2niix.configure()` to tell PGL where its executable is.

In [2]:
# Check that dcm2niix is available and record its version.
pglDcm2niix.validate()


Tool configuration: /Users/justin/.pgl/tools.json
* = default installation; cached = previous check, not fresh validation.

Tool      Installation  Status     Check  Version        Location
dcm2niix  default *     Available  fresh  v1.0.20261029  /Users/justin/miniforge3/envs/pgl_mri/bin/dcm2niix


True

In [3]:
# If the check reports that dcm2niix is unavailable, install it with Conda.
dcm2niix = pglDcm2niix()
dcm2niix.install()

+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=

IMPORTANT — AFTER INSTALLING dcm2niix

After installation completes successfully:
  1. Restart your notebook kernel.
  2. Run the following command to validate the installation and refresh its saved status:

from pgl.pglExternalTools import pglDcm2niix
pglDcm2niix.validate(timeout=120)

+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=
(pglDcm2niix:install) Installed and verified dcm2niix v1.0.20261029 at /Users/justin/miniforge3/envs/pgl_mri/bin/dcm2niix.


---

AFNI

---

In [ ]:
# Install AFNI. If already installed, will not try to re-install but will display information about installation
afni = pglAFNI()
#afni.configure()

In [ ]:
afni.install()

In [ ]:
afni.validate()

In [ ]:
afni.print()

---

GLMsingle

---

In [ ]:
GLMsingle = pglGLMsingle()
GLMsingle.configure()

In [ ]:

GLMsingle.install()

In [ ]:
available = pglGLMsingle.validate(timeout=120)

---

MATLAB

---

In [16]:
matlab = pglMATLAB()

In [17]:
matlab.startSession()

In [18]:
matlab.showDesktop()

In [ ]:
matlab.configure()

In [ ]:
matlab.install()

In [ ]:
pglDatabase.print(experimentName='nsd-mini')

In [ ]:
session = pglDatabase.loadSession(136)

In [ ]:
session.mri.viewer(scanNum=6, backend="mlrvol")
#session.mri.print()

In [ ]:
################################################################
# Create a synthetic NIfTI and open it in the existing MATLAB session
# No acquisition data is used. All generated files are retained.
################################################################
from pathlib import Path
from uuid import uuid4

import nibabel as nib
import numpy as np

if globals().get("matlab") is None:
    raise RuntimeError("Create your configured pglMATLAB instance as matlab first.")

testDirectory = Path.home() / "Desktop" / ("pgl_nifti_test_" + uuid4().hex[:8])
testDirectory.mkdir(parents=True, exist_ok=False)
testImagePath = testDirectory / "synthetic_phantom.nii.gz"

imageShape = (96, 112, 80)
voxelSizeMM = 2.0

coordinates = [
    (np.arange(size, dtype=np.float32) - (size - 1) / 2) * voxelSizeMM
    for size in imageShape
]
x, y, z = np.meshgrid(*coordinates, indexing="ij", sparse=True)

radius = (x / 75) ** 2 + (y / 90) ** 2 + (z / 60) ** 2
imageData = np.zeros(imageShape, dtype=np.float32)
imageData[radius < 1.0] = 70
imageData[radius < 0.75] = 110

# Asymmetric features make the views easier to distinguish.
imageData[(x / 15) ** 2 + (y / 28) ** 2 + (z / 18) ** 2 < 1] = 20
imageData[(x - 30) ** 2 + (y + 20) ** 2 + (z - 10) ** 2 < 12 ** 2] = 220
imageData[(x + 25) ** 2 + (y - 35) ** 2 + (z + 15) ** 2 < 8 ** 2] = 160

affine = np.diag([voxelSizeMM, voxelSizeMM, voxelSizeMM, 1.0])
affine[:3, 3] = -(np.asarray(imageShape) - 1) * voxelSizeMM / 2

image = nib.Nifti1Image(imageData, affine)
image.set_qform(affine, code=1)
image.set_sform(affine, code=1)
image.header.set_xyzt_units("mm")
image.header["descrip"] = b"PGL synthetic viewer test; not acquisition data"
image.header["cal_min"] = 0
image.header["cal_max"] = 220
nib.save(image, testImagePath)

# Check the saved file before viewing.
reloaded = nib.load(testImagePath)
assert reloaded.shape == imageShape
np.testing.assert_allclose(reloaded.affine, affine)
np.testing.assert_array_equal(np.asarray(reloaded.dataobj), imageData)

print(f"Synthetic NIfTI: {testImagePath}")
print("mlrVol:", matlab.feval("which", "mlrVol"))

viewerResult = matlab.viewer(
    testImagePath,
    cacheDirectory=testDirectory,
)

print(f"Independent viewer copy: {viewerResult['viewerImage']}")